# Weather Hazard Category · MLP Classifier

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Experiment notebook · predicts WHC exactly +7 days ahead*

This notebook develops the **MLP Classifier** model for the **Weather Hazard Category (WHC)**: pipeline stages 1-2, using the leakage-audited feature matrix produced by the preprocessing notebook.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Hypothesis

A small multi-layer perceptron will **not beat the tree ensembles or the linear model** on macro
F1: the minority classes offer too few examples for a network to learn a boundary around them,
and `MLPClassifier` has no class-weighting option to compensate.

That limitation is the reason to run the experiment: it isolates how much of any ensemble's
advantage came from class weighting rather than from the model family.

### Design

| Choice | Value | Reason |
|:--|:--|:--|
| Algorithm | `StandardScaler` → `MLPClassifier` | Smooth decision boundaries; scaling stays inside the fit |
| Imbalance | **No mechanism available** | `MLPClassifier` has no `class_weight`; a stated limitation, not an oversight |
| Early stopping | 15% of the *training* data, 20 rounds patience | The project's validation split is never used for fitting |
| Strategy | One classifier for the single +7 day horizon | Matches the deployed artefact layout |
| Selection | Optuna on walk-forward CV, all features | Section 5 |
| Objective | validation **macro F1** (higher is better), averaged over the walk-forward folds | Used for every comparison |

### Where this notebook sits in the pipeline

| Stage | What happens | Where |
|:-:|:--|:--|
| 1–2 | Baselines, ablation and hyperparameter tuning on all features | **This notebook** |
| 3 | Shortlist the best tuned algorithms | Comparison notebook |
| 4 | Feature selection for the shortlist, then a short re-tune | Comparison notebook |
| 5–6 | Choose the champion; refit it on all development data | Comparison notebook |
| 7 | Score the champion once on the test split (2025) | Comparison notebook |
| 8 | Deploy with drift monitoring | Promotion + API |

Every score in this notebook is a **walk-forward cross-validation** average: the development split
(2016–2024) is divided into four expanding-window folds that validate on 2021, 2022, 2023 and 2024
in turn, with the embargo as a gap. A setting therefore has to work across four years, not one. The
test split is never loaded here.

### Workflow

All ten experiment notebooks run the same code from `nucleus_ml.weather.Experiment`, so their results are directly comparable. **Only the model specification in section 1 differs between notebooks.**

| # | Step | Purpose |
|:-:|:--|:--|
| 1 | Model specification | The only algorithm-specific cell |
| 2 | Data and folds | The development split and its walk-forward folds |
| 3 | Baselines | Stage 1: the floor every model must clear |
| 4 | Feature-family ablation | What each block of engineered features contributes |
| 5 | Hyperparameter tuning | Stage 2: Optuna, scored by walk-forward CV on all features |
| 6 | Cross-validated results | The model against its default and the baselines |
| 7 | Feature importance | Which features the model relies on |
| 8 | Generalisation check | Training vs validation performance |
| 9 | Save the experiment bundle | Model, CV results and specification |
| 10 | Conclusions | What to carry into the report |

---
## 0. Setup

Imports, the experiment's identity and the task definition. Everything else comes from the project package.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from nucleus_ml import CategoricalParameter, FloatParameter, IntParameter
from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

TASK = "weather_hazard"
EXPERIMENT = "whc_mlp"
ALGORITHM = "MLP Classifier"
VERBOSE = 1  # 0 = results only, 1 = progress + every Optuna trial, 2 = every model fit
N_TRIALS = 20  # Optuna trials per target (each trial fits one model per fold)

task = wx.load_task(TASK)
print(f"{task.target_name} | {ALGORITHM} | {len(task.feature_columns)} features | "
      f"targets: {task.target_columns}")

---
## 1. Model specification

This is the **only algorithm-specific cell**. It creates the experiment from a package builder function (`build=wx.…`), because this model's construction needs custom logic. It also sets `default_params` (the untuned reference), a cheaper `ablation_params` and the `search_space`, written with the `nucleus_ml` parameter classes.

The model is built by `wx.build_mlp_classifier` (in the package, so the comparison notebook can rebuild it):
a scaler followed by the network, with the layer layout searched as two numbers (`n_layers` and
`units`) that become `hidden_layer_sizes`, because Optuna cannot search a tuple directly.
Oversampling the rare days is the usual workaround for missing class weights, but
duplicating a few dozen days across 2,900 rows invites memorisation, so it is not applied.
`alpha` does double duty: heavy regularisation also pushes the network toward the majority
class.

In [ ]:
DEFAULT_PARAMS = {"n_layers": 2, "units": 128, "alpha": 1e-3, "learning_rate_init": 1e-3,
                  "activation": "relu"}

exp = wx.Experiment(
    task,
    name=EXPERIMENT,
    algorithm=ALGORITHM,
    verbose=VERBOSE,
    build=wx.build_mlp_classifier,
    default_params=DEFAULT_PARAMS,
    ablation_params={**DEFAULT_PARAMS, "n_layers": 1, "units": 32},
    search_space={
        "n_layers": IntParameter(1, 3),
        "units": IntParameter(16, 256, log=True),
        "alpha": FloatParameter(1e-6, 1e-1, log=True),
        "learning_rate_init": FloatParameter(1e-4, 2e-2, log=True),
        "activation": CategoricalParameter(["relu", "tanh"]),
    },
)

---
## 2. Data and folds

Only the **development split** (`train`, 2016–2024) is loaded. The table lists the
walk-forward folds: each trains on every year before its validation year, with the
embargo between them.

> The preprocessing notebook already applied the chronological split, the warm-up
> filter and the embargo before the test period. Nothing is shuffled here.

In [ ]:
exp.data_summary()

---
## 3. Baselines

Three naive forecasts, refitted on the training years of every fold and scored on its
validation year (the table shows the mean over the folds):

| Baseline | Rule | What it knows |
|:--|:--|:--|
| **Majority class** | Always predict the most common class | Nothing but the class balance |
| **Persistence** | The class in 7 days equals today's class (from today's WHI) | The current weather |
| **Seasonal mode** | The most common class for that month | The season |

The majority-class rule scores a high *accuracy* but a low *macro F1*, which is exactly why
accuracy is not used to select models for this imbalanced target.

In [ ]:
exp.run_baselines()

---
## 4. Feature-family ablation

Feature families are added **one at a time**, starting from persistence alone, and the model is cross-validated at each step with `ablation_params`. This *measures* what each block of engineering contributes; choosing features happens later, in the comparison notebook. Scores are mean macro F1 over the folds.

In [ ]:
ablation = exp.run_ablation()
plt.show()
ablation

---
## 5. Hyperparameter tuning

Optuna (seeded TPE sampler) searches the space defined in section 1, **on all
features**. Every trial is cross-validated on the four folds and scored by its
mean, so the search cannot overfit to one year. Each target is tuned separately.
The plots show the search history and which settings mattered most.

In [ ]:
exp.tune(n_trials=N_TRIALS)

In [ ]:
exp.plot_search()
plt.show()

---
## 6. Cross-validated results

The chosen configuration is cross-validated again and compared with the untuned `default` and every baseline, all on the same folds. `gain_of_model` is positive when the model beats that row. The plot shows the **out-of-fold predictions**: every validation year predicted by a model that never saw it.

Read the **confusion matrix** as well as the headline score: a model that almost never predicts
the rare classes has not learned to flag hazardous days, whatever its macro F1. The printed
boundary summary shows how many errors sit within a few WHI points of a class threshold; those are
threshold effects rather than genuine misses.

In [ ]:
exp.assess()

In [ ]:
exp.plot_validation()
plt.show()

---
## 7. Feature importance

This model has no built-in importance, so **permutation importance** is used: each feature is shuffled on the validation split and the drop in score is recorded. Features with negative importance are candidates for pruning.

> Importance is measured on the last fold's model and validation year (2024). Correlated features share importance, so the ablation in section 4 is the better guide to what can be removed.

In [ ]:
family_importance = exp.importance()
plt.show()
family_importance

---
## 8. Generalisation check

Mean training and validation scores over the folds. A small gap means the model
generalises; a large gap means it memorises the training years.

In [ ]:
exp.generalisation()

---
## 9. Save the experiment bundle

The chosen configuration is refitted on **all development data** and saved to
`models/weather_hazard/experiments/whc_mlp/`:

| File | Contents |
|:--|:--|
| `whc_mlp_h<horizon>.joblib` | The model, one per horizon |
| `metadata.json` | Settings, features, importance and the cross-validated scores |
| `manifest.json` | File list and the feature column order |
| `spec.joblib` | The specification and tuned settings, which the comparison notebook uses to rebuild this experiment |

In [ ]:
exp.export()

---
## 10. Conclusions

- **The class-weighting comparison.** If the unweighted MLP matches the weighted ensembles,
  weighting was not what mattered; if it collapses onto the majority class, it was essential.
- **Predicted class distribution.** Check the per-class table first: a network that predicts one
  class every day has a low macro F1 however it is tuned.
- **Cost against benefit.** The MLP is the least interpretable and most expensive option; it needs
  a clear advantage over the logistic regression to justify deployment.

> If this algorithm is shortlisted, the comparison notebook selects its feature families with walk-forward cross-validation and briefly re-tunes its settings on the chosen features.

> The test split has still not been used. It is scored exactly once, in the `whc_comparison` notebook, after the champion has been chosen.